# 04 — Dataset, synchronized augmentation, sampler

**What this notebook does.** Turns the fold manifests into a `torch` dataset:
tiles cropped on the fly from the full images at the manifest's coordinates,
spatial augmentation applied identically to image and mask, photometric
augmentation applied to the image alone, and a per-image z-score. It then
proves the parts that fail silently — that masks stay binary, that image and
mask move together — by running the test suite and by showing you what broken
synchronization looks like.

**What must already exist.**

- `GH_TOKEN` as a host secret and the datasets mounted — `00_bootstrap.ipynb`
- `reports/manifests/*.csv` and `configs/fold_stats.yaml` — `03_tiling.ipynb`
- `reports/gt_extraction.json` and the boundary PNGs — `02_boundary_gt.ipynb`

**What it produces.** `configs/dataloader.yaml` — the measured best
`num_workers` for this host — and the confidence that the training step can be
written against `src/dataset.py`. No image data is written.

**Expected runtime on a free T4.** 5–9 minutes, almost all of it the
`num_workers` timing sweep. No GPU is used: this step is pure I/O and CPU.

## Cell 1 — the standard bootstrap block

Identical in every notebook. Reads `GH_TOKEN` from the host secret store,
fetches `scripts/bootstrap_session.py`, then hands over to `bootstrap()`,
which syncs the repo, installs what is missing, mounts Drive on Colab and
returns `PATHS`.

In [ ]:
# --- standard bootstrap block: identical in every notebook ---------------
OWNER, REPO, BRANCH = "arhorri", "boundary", "main"

import importlib, os, pathlib, sys, urllib.request


def _gh_token():
    """Read GH_TOKEN from whichever secret store this host provides."""
    try:
        from google.colab import userdata

        return userdata.get("GH_TOKEN")
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret("GH_TOKEN")
    except Exception:
        pass
    return os.environ.get("GH_TOKEN")


_token = _gh_token()
if not _token:
    raise SystemExit(
        "GH_TOKEN secret is missing.\n"
        "  Colab : key icon in the left sidebar -> add GH_TOKEN -> notebook access ON\n"
        "  Kaggle: Add-ons -> Secrets -> add GH_TOKEN -> attach to this notebook"
    )

_req = urllib.request.Request(
    f"https://api.github.com/repos/{OWNER}/{REPO}/contents/scripts/bootstrap_session.py?ref={BRANCH}",
    headers={
        "Authorization": f"Bearer {_token}",
        "Accept": "application/vnd.github.raw",
    },
)
pathlib.Path("bootstrap_session.py").write_bytes(urllib.request.urlopen(_req).read())
del _token

if str(pathlib.Path.cwd()) not in sys.path:
    sys.path.insert(0, str(pathlib.Path.cwd()))
import bootstrap_session

bootstrap_session = importlib.reload(bootstrap_session)

PATHS = bootstrap_session.bootstrap(
    repo_url=f"https://github.com/{OWNER}/{REPO}.git", branch=BRANCH
)

## Why spatial and photometric augmentation are separated

**The mask must move, but must never be edited.** A spatial transform — flip,
rotation, small affine, mild elastic — changes *where* things are. If the
image is rotated 12° and the mask is not, every label is wrong by 12°, the
loss is minimised by predicting nothing, and no error is ever raised. So
spatial transforms are one `Compose` applied to both targets at once, sharing
a single sampled parameter set.

Photometric transforms — brightness, gamma, noise, blur, CLAHE — change *what
a pixel is worth*. Applied to a mask they are nonsense: a blurred mask has
values like 0.37, which is neither boundary nor background, and a brightened
mask is simply a different label. So the mask is **not passed to the
photometric pipeline at all**. It cannot be corrupted by a transform it never
reaches — that is a structural guarantee, not a configuration flag.

Interpolation follows the same logic. The image is resampled bilinearly
because intermediate intensities are meaningful; the mask is resampled
nearest-neighbour because intermediate labels are not. `verify_interpolation()`
walks the built pipeline and checks the attributes that actually ended up on
the objects, and `assert_binary()` checks the values of every sample that
leaves `__getitem__` — configuration and outcome, both.

**Why photometric variety matters more here than usual.** These five datasets
span SEM and optical microscopy, grayscale and colour, different etches,
magnifications and detectors. Every fold holds out a *different microscope*,
and this pipeline has no domain-adaptation stage. Photometric augmentation is
the only preparation the model gets for an imaging setup it has never seen:
brightness, gamma, noise, blur and local contrast are precisely the axes along
which two microscopes disagree about the same specimen.

**Why normalization is per-image.** Steel2 is optical colour; the other four
are grayscale SEM. There is no mean and standard deviation that is correct for
both, and a shared constant would bake the domain difference into the input —
the very difference the held-out fold exists to measure. Each tile is
z-scored against itself, computed after augmentation so that train and val
tiles reach the network with the same first two moments.

## Build the dev fold

`dev` is the alias step 3 gave to the fold with the smallest validation set —
same protocol, faster iteration. The train split gets augmentation; val
returns the raw tile, deterministically, because a validation number that
changes between runs cannot be compared.

The image cache is filled here, and it has two speeds. A **cold** build
decodes ~1,000 source images off Drive at roughly 0.6 s each — that is
round-trip latency on small files, not decode time, and it adds up to about 27
minutes. A **warm** load reads one file containing every decoded array and
takes seconds. The cache is keyed by a hash of the source images and the crops
applied to them, so a changed manifest rebuilds instead of silently serving
arrays that no longer match; the key, the outcome and the reason are all
printed.

The cell then loads the same fold a second time to measure the warm path in
this same run, rather than asking you to take the speedup on trust.

The sample printout below is worth reading closely: shapes, dtypes, the value
range after normalization, and the `dataset` / `parent_id` that travel with
every sample.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from src import dataset as ds

settings = ds.load_config()
reports_dir = Path(PATHS["reports_dir"])
manifest_dir = reports_dir / settings["manifest_subdir"]
FOLD = "dev"
manifest = manifest_dir / f"{FOLD}.csv"

roots = {"data_root": Path(PATHS["data_root"]),
         "gt_root": Path(PATHS["gt_boundaries_root"])}
crops = ds.load_crops(reports_dir)
print(f"manifest: {manifest}")
print(f"recorded crops reused from step 2: "
      f"{[f'{k[0]}/{k[1]}' for k in crops] or 'none'}\n")

train_ds = ds.TileDataset.from_manifest(manifest, "train", settings=settings,
                                        crops=crops, roots=roots)
val_ds = ds.TileDataset.from_manifest(manifest, "val", settings=settings,
                                      crops=crops, roots=roots)

print(f"train tiles {len(train_ds)}  (augment={train_ds.augment})")
print(f"val   tiles {len(val_ds)}  (augment={val_ds.augment})")

# Fill the image cache, from one file if it exists and from ~1000 Drive reads
# if it does not. The cold path costs ~0.6 s per image -- round-trip latency,
# not decode time -- which is ~27 minutes, paid again on every session start
# and every crash resume. The warm path is a single sequential read.
cache_dir = Path(PATHS["persistent_dir"]) / settings["cache_subdir"]
cache_stats = {}
if settings["cache_preload"]:
    from tqdm.auto import tqdm

    for name, split_ds in (("train", train_ds), ("val", val_ds)):
        estimate = split_ds.cache_estimate()
        print(f"\n{name}: {estimate['images']} source images, "
              f"{estimate['mb']:.1f} MB estimated "
              f"(cap {settings['cache_max_mb']} MB)")
        for dataset_name, info in estimate["per_dataset"].items():
            print(f"    {dataset_name:<10} {info['images']:>4} images  "
                  f"{info['mb']:>7.1f} MB")
        stats_cache = split_ds.ensure_cache(
            cache_dir, name=f"{FOLD}-{name}",
            progress=lambda seq: tqdm(seq, desc=f"decoding {name}", leave=False,
                                      unit="img"))
        cache_stats[name] = stats_cache
        print(f"    {stats_cache['source'].upper()}: {stats_cache['reason']}")
        print(f"    {stats_cache['images']} images, {stats_cache['mb']:.1f} MB "
              f"held, {stats_cache['seconds']:.1f} s")
        if stats_cache["written"]:
            print(f"    wrote {stats_cache['written']['mb']:.1f} MB to "
                  f"{Path(stats_cache['written']['blob']).name}")

    # Time a SECOND load in this same run: a fresh dataset over the same rows
    # must come back warm, from the file just written. This is the measurement
    # of the fix, made here rather than promised.
    reload_ds = ds.TileDataset(train_ds.rows, settings=settings, augment=True,
                               crops=crops, roots=roots)
    warm = reload_ds.ensure_cache(cache_dir, name=f"{FOLD}-train")
    cache_stats["warm_reload"] = warm
    print(f"\nsecond load of the same fold: {warm['source'].upper()} in "
          f"{warm['seconds']:.1f} s ({warm['reason']})")
    cold = cache_stats["train"]
    if cold["source"] == "cold":
        print(f"    cold build was {cold['seconds']:.1f} s -> "
              f"{cold['seconds'] / max(0.01, warm['seconds']):.0f}x faster warm")
    del reload_ds

interp = ds.verify_interpolation(train_ds.spatial)
print(f"\ninterpolation verified on: {interp['verified']}")
if interp["unverified"]:
    print(f"not structurally verifiable on this albumentations version: "
          f"{interp['unverified']} — covered by the per-sample binary check")
if interp["unset"]:
    print(f"containers deferring to their children (no forced override): "
          f"{interp['unset']}")
print(f"border fill REFLECT_101 verified on: {interp['border_verified']}")
if interp["border_unverified"]:
    print(f"border mode not inspectable on: {interp['border_unverified']} — "
          f"covered by the constant-fill check below")

sample = train_ds[0]
print("\none augmented train sample")
for key in ("image", "mask"):
    arr = sample[key]
    print(f"  {key:<6} shape {arr.shape}  dtype {arr.dtype}  "
          f"min {arr.min():+.4f}  max {arr.max():+.4f}  "
          f"mean {arr.mean():+.4f}  std {arr.std():.4f}")
print(f"  unique mask values: {np.unique(sample['mask']).tolist()}")
for key in ("dataset", "parent_id", "tile_id"):
    print(f"  {key:<10} {sample[key]!r}")

## What the validation split is actually made of

Every fold's validation set is deliberately mixed: the held-out dataset plus
Steel1's four validation parents. That is by design — Steel1 appears on both
sides of every fold because it is 91% of the tiles and cannot be spent — but
it has a consequence for step 6.

**Validation metrics must be reported per dataset, never pooled.** Steel1's
boundaries are large, blob-like and easy; the held-out dataset is a different
microscope and is the only thing the fold actually measures. A pooled Dice
would let Steel1's easy tiles carry the average while the model fails on the
held-out set, and the number would look fine. The `dataset` field on every
sample exists so that grouping is possible at all.

In [ ]:
rows = []
for name, split_ds in (("train", train_ds), ("val", val_ds)):
    for dataset_name, info in split_ds.composition().items():
        rows.append({"split": name, "dataset": dataset_name,
                     "tiles": info["tiles"], "parents": info["parents"]})
composition = pd.DataFrame(rows)
display(composition.pivot_table(index="dataset", columns="split",
                                values=["tiles", "parents"], fill_value=0))

val_comp = val_ds.composition()
total = sum(v["tiles"] for v in val_comp.values())
print(f"\nval split composition ({total} tiles):")
for dataset_name, info in val_comp.items():
    print(f"  {dataset_name:<10} {info['tiles']:>5} tiles "
          f"({info['tiles'] / total:>5.1%})  {info['parents']} parents")
print("\nStep 6 must report metrics per dataset within val, not pooled: the "
      "held-out dataset is what this fold measures.")

## Run the test suite

The tests are the part of this step that actually proves anything, and they
run here because there is no local environment to run them in. They check the
properties that fail silently rather than loudly: masks binary across many
random seeds, image and mask receiving the same geometry, the reconciled
MetalDam pairs loading at their cropped size, and a stale padded manifest
being refused.

A skipped test is not a passing test — the checks cell below treats skips as
failures when they mean the data was not reachable.

In [ ]:
import subprocess
import sys

pytest_run = subprocess.run(
    [sys.executable, "-m", "pytest", "-v", "--tb=short", "-p", "no:cacheprovider",
     str(Path(PATHS["repo_root"]) / "tests" / "test_dataset.py")],
    cwd=str(PATHS["repo_root"]), capture_output=True, text=True)

print(pytest_run.stdout[-6000:])
if pytest_run.stderr.strip():
    print("stderr:", pytest_run.stderr[-2000:])
print(f"pytest exit code: {pytest_run.returncode}")

## Augmented pairs, with the mask drawn over the image

Eight augmented samples, the mask overlaid in red at partial opacity. This is
the fastest way to see a synchronization failure: if the spatial transform
were applied to only one of the two, the red lines would sit beside the
boundaries in the image rather than on them.

Look along the boundaries, not at the picture as a whole. The red should trace
the same ridges the eye follows in the grayscale underneath, in every tile,
including the rotated and elastically warped ones.

**No black wedges.** Geometric transforms default to filling vacated corners
with a constant, which paints a dead-straight, high-contrast seam into the
tile — and the mask there reads 0, so it would teach a *boundary detector*
that strong straight edges are not boundaries. Every geometric op therefore
uses `BORDER_REFLECT_101`. That is not the tile-edge padding removed in step
3: that was a systematic 48% of specific tiles fabricated identically every
epoch at a fixed place, while this is a small wedge that moves with each
random draw, is real texture from the same specimen, and carries the mask with
it — a reflected boundary is still labelled a boundary. The `fill` figure in
each panel title is the largest exactly-uniform region touching that tile's
border, measured on the produced tile; it should read 0.000.

In [ ]:
import matplotlib.pyplot as plt

N_SHOW = 8
rng = np.random.default_rng(0)
picks = rng.choice(len(train_ds), size=min(N_SHOW, len(train_ds)), replace=False)

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax, idx in zip(axes.ravel(), picks):
    s = train_ds[int(idx)]
    image = s["image"][0]
    mask = s["mask"][0]
    shown = (image - image.min()) / max(1e-6, float(np.ptp(image)))
    rgb = np.dstack([shown] * 3)
    rgb[..., 0] = np.where(mask > 0, 1.0, rgb[..., 0])
    rgb[..., 1] = np.where(mask > 0, rgb[..., 1] * 0.45, rgb[..., 1])
    rgb[..., 2] = np.where(mask > 0, rgb[..., 2] * 0.45, rgb[..., 2])
    ax.imshow(rgb, interpolation="nearest")
    fill = ds.constant_border_region(image)
    ax.set_title(f"{s['dataset']} · {s['parent_id'][:24]}\n"
                 f"boundary {mask.mean():.3f} · fill {fill['fraction']:.3f}",
                 fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle("augmented train tiles — mask in red; 'fill' is the largest "
             "uniform border-touching region (0.000 = no fabricated corner)",
             fontsize=13)
plt.tight_layout()
plt.show()

## Negative control: what broken synchronization looks like

The panels above only mean something if you know what the failure looks like,
so this cell creates one deliberately. The left pair is correct: image and
mask receive the same rotation. The right pair is broken: the image is rotated
and the mask is not.

The broken version is what you get from applying a transform to `image=` and
forgetting to pass `mask=`, or from building two separate transforms with
independently sampled parameters. Note how ordinary it looks at a glance —
the tile is a plausible micrograph and the mask is a plausible boundary
network. Only the overlay reveals that they describe different geometry, and
in training there is no overlay: the loss simply learns to predict the average
of everything.

In [ ]:
import albumentations as A
import cv2


def overlay(image, mask):
    shown = (image - image.min()) / max(1e-6, float(np.ptp(image)))
    rgb = np.dstack([shown] * 3)
    rgb[..., 0] = np.where(mask > 0, 1.0, rgb[..., 0])
    rgb[..., 1] = np.where(mask > 0, rgb[..., 1] * 0.4, rgb[..., 1])
    rgb[..., 2] = np.where(mask > 0, rgb[..., 2] * 0.4, rgb[..., 2])
    return rgb


raw = ds.TileDataset(val_ds.rows[:1], settings=settings, augment=False,
                     crops=crops, roots=roots)[0]
image, mask = raw["image"][0], raw["mask"][0]

rotate = A.Affine(rotate=(20, 20), p=1.0, interpolation=cv2.INTER_LINEAR,
                  mask_interpolation=cv2.INTER_NEAREST)
correct = rotate(image=image, mask=mask)
broken_image = rotate(image=image)["image"]   # the mask is NOT transformed

fig, axes = plt.subplots(1, 3, figsize=(17, 6))
axes[0].imshow(overlay(image, mask), interpolation="nearest")
axes[0].set_title("original tile\nmask on the image", fontsize=10)
axes[1].imshow(overlay(correct["image"], correct["mask"]), interpolation="nearest")
axes[1].set_title("CORRECT — image and mask rotated together\n"
                  "red still traces the boundaries", fontsize=10)
axes[2].imshow(overlay(broken_image, mask), interpolation="nearest")
axes[2].set_title("BROKEN — image rotated, mask left behind\n"
                  "red now labels the wrong pixels", fontsize=10)
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

agreement = float((correct["mask"] > 0).astype(np.float32).flatten() @
                  (mask > 0).astype(np.float32).flatten())
print(f"overlap between the correctly-rotated mask and the un-rotated one: "
      f"{agreement:.0f} pixels — this is the error the loss would be asked to fit")

## The two reconciled MetalDam pairs

`micrograph15` and `micrograph19` have masks one row shorter than their
images. Step 2 recorded the exact crop it applied; this step **reads that
crop** from `reports/gt_extraction.json` rather than recomputing it, because a
crop derived twice can differ by a row and misalign every label in the image
by one pixel — an error too small to see and large enough to matter for a 2 px
boundary.

`read_pair()` also asserts that the cropped image and the boundary map have
identical shape, and raises with both shapes if not.

In [ ]:
shown = 0
for (name, image_name), crop in crops.items():
    matching = [r for r in train_ds.rows + val_ds.rows
                if r["dataset"] == name and r["source_image"] == image_name]
    if not matching:
        print(f"{name}/{image_name}: recorded crop {crop}, but no tile of it is "
              f"in the {FOLD} fold")
        continue
    row = matching[0]
    image, gt = ds.read_pair(row, crops=crops, roots=roots)
    sample = ds.TileDataset([row], settings=settings, augment=False,
                            crops=crops, roots=roots)[0]
    print(f"{name}/{image_name}")
    print(f"    recorded crop      left {crop['left']}, top {crop['top']}, "
          f"{crop['width']}x{crop['height']}")
    print(f"    image after crop   {image.shape[1]}x{image.shape[0]}")
    print(f"    boundary map       {gt.shape[1]}x{gt.shape[0]}  "
          f"(identical: {image.shape == gt.shape})")
    print(f"    tile {row['tile_id']} -> image {tuple(sample['image'].shape)}, "
          f"mask {tuple(sample['mask'].shape)}")
    shown += 1
print(f"\n{shown} reconciled pair(s) loaded at the cropped size")

## How many workers does this host want — and does the cache help?

Two things are measured here, because they interact.

**Drive I/O is the real bottleneck, not the CPU.** Between two runs on the
same host, same data, same batch count, single-threaded throughput fell from
115.7 to 4.8 tiles/s — a 24x collapse that no augmentation change can explain.
Every tile is a crop of a full image stored on Drive, so the loader's speed is
Drive's speed that day. At 4.8 tiles/s a 2,333-tile epoch is eight minutes of
pure reading on a GPU that should be waiting milliseconds.

The cache removes that variance: each source image is decoded once, up front,
and every later epoch reads RAM. The sweep below therefore runs **twice** —
once cached, once uncached — and records both. Keeping the uncached numbers is
the point: they are the measurement of how bad the I/O is, and hiding them
behind the fix would make the next regression invisible.

**Workers are not free once there is a cache.** Each worker process inherits
the parent's cache by fork. Large NumPy buffers stay shared copy-on-write, but
the Python objects around them do not, so N workers costs somewhat more than
one copy — and on a 2-CPU box the extra processes may not even pay for
themselves. With the data in RAM the loader may well be fastest at
`num_workers=0`, which also silences the oversubscription warning. That is why
the winner is chosen from the measurement rather than assumed.

In [ ]:
import time

import yaml
from torch.utils.data import DataLoader

BATCHES = 200
BATCH_SIZE = 8
WARMUP = 5
CANDIDATES = [0, 2, 4]


def time_loader(dataset, workers, batches=BATCHES):
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=workers, drop_last=True,
                        persistent_workers=bool(workers))
    it = iter(loader)
    for _ in range(WARMUP):
        try:
            next(it)
        except StopIteration:
            it = iter(loader)
            next(it)
    seen, start = 0, time.perf_counter()
    while seen < batches:
        try:
            next(it)
        except StopIteration:
            it = iter(loader)
            next(it)
        seen += 1
    elapsed = time.perf_counter() - start
    del loader, it
    return elapsed


# The cached dataset is train_ds, already preloaded above. The uncached one
# reads from Drive on every access -- the situation before this fix, kept as a
# live measurement of Drive variance rather than a remembered number.
uncached_ds = ds.TileDataset(train_ds.rows, settings=settings, augment=True,
                             crops=crops, roots=roots, cache_images=False)

UNCACHED_BATCHES = 25   # enough to measure; a full sweep uncached is minutes
results = {"cached": {}, "uncached": {}}

print(f"cached ({train_ds.cache_bytes() / 1024 ** 2:.1f} MB held):")
for workers in CANDIDATES:
    elapsed = time_loader(train_ds, workers)
    results["cached"][workers] = BATCHES * BATCH_SIZE / elapsed
    print(f"  num_workers={workers}: {elapsed:6.2f} s for {BATCHES} batches "
          f"({results['cached'][workers]:8.1f} tiles/s)")

print(f"\nuncached (reads Drive per tile, {UNCACHED_BATCHES} batches each):")
for workers in CANDIDATES:
    elapsed = time_loader(uncached_ds, workers, batches=UNCACHED_BATCHES)
    results["uncached"][workers] = UNCACHED_BATCHES * BATCH_SIZE / elapsed
    print(f"  num_workers={workers}: {elapsed:6.2f} s for {UNCACHED_BATCHES} "
          f"batches ({results['uncached'][workers]:8.1f} tiles/s)")

best = max(results["cached"], key=results["cached"].get)
speedup = results["cached"][best] / max(1e-9, results["uncached"][best])
print(f"\nbest cached: num_workers={best} "
      f"({results['cached'][best]:.1f} tiles/s), "
      f"{speedup:.1f}x the uncached rate at the same worker count")
epoch_s = len(train_ds) / results["cached"][best]
print(f"one {len(train_ds)}-tile epoch of pure loading: {epoch_s:.1f} s cached, "
      f"{len(train_ds) / max(1e-9, results['uncached'][best]):.1f} s uncached")

# The value is HOST-SCOPED, not a single number: Colab gives 4 CPUs and
# Kaggle 2, so one measured num_workers is wrong on the other host. Each
# platform writes its own entry and leaves the others untouched.
dataloader_path = Path(PATHS["repo_root"]) / "configs" / "dataloader.yaml"
existing = {}
if dataloader_path.is_file():
    existing = yaml.safe_load(dataloader_path.read_text()) or {}
hosts = dict(existing.get("hosts") or {})

platform = PATHS["platform"]
hosts[platform] = {
    "measured_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "gpu": PATHS["gpu"],
    "cpu_count": os.cpu_count(),
    "batch_size": BATCH_SIZE,
    "batches_timed": BATCHES,
    "num_workers": int(best),
    "cache_images": True,
    "cache_mb": round(train_ds.cache_bytes() / 1024 ** 2, 1),
    "cached_tiles_per_second": {int(k): round(v, 1)
                                for k, v in results["cached"].items()},
    "uncached_tiles_per_second": {int(k): round(v, 1)
                                  for k, v in results["uncached"].items()},
    # Cold = decode every source image off Drive; warm = one sequential read
    # of the persistent cache. Both are kept so the next regression in either
    # is visible instead of being absorbed by the other.
    "cache_cold_seconds": cache_stats.get("train", {}).get("seconds"),
    "cache_cold_source": cache_stats.get("train", {}).get("source"),
    "cache_warm_seconds": cache_stats.get("warm_reload", {}).get("seconds"),
    "cache_key": cache_stats.get("train", {}).get("key"),
    "cache_dir": str(cache_dir),
}
dataloader_path.write_text(
    "# DataLoader settings measured on each host by notebooks/04_dataset.ipynb.\n"
    "# Keyed BY PLATFORM: Colab and Kaggle have different CPU counts, so a\n"
    "# single num_workers would be wrong on one of them. Running this notebook\n"
    "# on a host updates only that host's entry.\n"
    "#\n"
    "# Read it as: hosts[<platform>][num_workers], falling back to 2.\n"
    "#\n"
    "# Both cached and uncached rates are recorded on purpose. The uncached\n"
    "# numbers measure Drive I/O, which has varied 24x between sessions on\n"
    "# identical hardware; keeping them visible is how that stays diagnosable\n"
    "# instead of being hidden behind the cache.\n"
    + yaml.safe_dump({"hosts": hosts}, sort_keys=False))
print(f"\nwrote {dataloader_path} (host '{platform}')")
for name, entry in hosts.items():
    marker = " <- this host" if name == platform else ""
    print(f"    {name:<8} num_workers={entry['num_workers']}  "
          f"cpus={entry.get('cpu_count')}{marker}")

## Checks

Where the step is declared correct or not. These re-verify the properties the
tests cover, but on the real fold and through a real `DataLoader`, because a
passing unit test on synthetic data and a working batch are different claims.

In [ ]:
checks = []


def check(name, ok, detail=""):
    checks.append((name, bool(ok)))
    print(f"{'PASS' if ok else 'FAIL'}  {name}{'  -- ' + detail if detail else ''}")


check("pytest suite passed", pytest_run.returncode == 0,
      f"exit code {pytest_run.returncode}")
skipped = "skipped" in pytest_run.stdout.lower()
check("no test skipped for missing data", not skipped,
      "a skip here means the data was unreachable, not that the test passed"
      if skipped else "none skipped")

SAMPLES = 32
picks = np.random.default_rng(1).choice(len(train_ds), size=SAMPLES, replace=False)
mask_values, means, stds, fields = set(), [], [], True
for idx in picks:
    s = train_ds[int(idx)]
    mask_values |= set(np.unique(s["mask"]).tolist())
    means.append(float(s["image"].mean()))
    stds.append(float(s["image"].std()))
    fields &= all(isinstance(s[k], str) and s[k]
                  for k in ("dataset", "parent_id", "tile_id"))

check("mask values strictly {0, 1}", mask_values <= {0.0, 1.0},
      f"observed {sorted(mask_values)} over {SAMPLES} augmented samples")
check("image mean ~ 0 after normalization", max(abs(m) for m in means) < 1e-3,
      f"max |mean| = {max(abs(m) for m in means):.2e}")
check("image std ~ 1 after normalization",
      all(abs(sd - 1.0) < 0.02 for sd in stds),
      f"std range {min(stds):.4f}-{max(stds):.4f}")
check("dataset / parent_id / tile_id present and non-empty on every sample",
      fields, f"{SAMPLES} samples")

# Constant-fill corners would be a fabricated straight edge labelled
# background: measured on the produced tiles, not inferred from the config.
FILL_LIMIT = 0.01
fills = [ds.constant_border_region(train_ds[int(i)]["image"])
         for i in np.random.default_rng(2).choice(len(train_ds), size=SAMPLES,
                                                  replace=False)]
worst_fill = max(f["fraction"] for f in fills)
check(f"no augmented tile has a uniform border region over {FILL_LIMIT:.0%}",
      worst_fill < FILL_LIMIT,
      f"largest uniform border-touching region {worst_fill:.4f} of the tile "
      f"over {SAMPLES} augmented samples")
check("border fill mode verified as REFLECT_101",
      bool(interp["border_verified"]) and not interp["border_unverified"],
      f"verified {interp['border_verified']}, unverifiable "
      f"{interp['border_unverified']}")

loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=0)
batch = next(iter(loader))
check("batch shapes correct",
      tuple(batch["image"].shape) == (4, 1, 256, 256)
      and tuple(batch["mask"].shape) == (4, 1, 256, 256),
      f"image {tuple(batch['image'].shape)}, mask {tuple(batch['mask'].shape)}")
check("batch dtypes float32",
      batch["image"].dtype == torch.float32 and batch["mask"].dtype == torch.float32,
      f"{batch['image'].dtype}, {batch['mask'].dtype}")
check("string fields survive collation",
      len(batch["dataset"]) == 4 and all(batch["dataset"]),
      f"{list(batch['dataset'])}")

reconciled_ok, reconciled_detail = True, []
for (name, image_name), crop in crops.items():
    matching = [r for r in train_ds.rows + val_ds.rows
                if r["dataset"] == name and r["source_image"] == image_name]
    if not matching:
        continue
    image, gt = ds.read_pair(matching[0], crops=crops, roots=roots)
    ok = image.shape == gt.shape == (crop["height"], crop["width"])
    reconciled_ok &= ok
    reconciled_detail.append(f"{image_name} {image.shape[1]}x{image.shape[0]}")
check("reconciled pairs load at the cropped size", reconciled_ok,
      "; ".join(reconciled_detail) or "none in this fold")

fold_stats = ds.load_fold_stats()
sampler = ds.build_sampler(train_ds, FOLD, fold_stats=fold_stats)
check("sampler built from fold_stats weights",
      len(sampler.weights) == len(train_ds),
      f"{len(sampler.weights)} weights, pos_weight "
      f"{ds.pos_weight(FOLD, fold_stats=fold_stats)}")

check("val split is deterministic (no augmentation)",
      np.array_equal(val_ds[0]["image"], val_ds[0]["image"]),
      f"augment={val_ds.augment}")
cache_estimate = train_ds.cache_estimate()
cache_mb = train_ds.cache_bytes() / 1024 ** 2
check(f"image cache under the {settings['cache_max_mb']} MB cap",
      cache_mb <= float(settings["cache_max_mb"]),
      f"{cache_mb:.1f} MB held for {len(train_ds._cache)} source images "
      f"(estimated {cache_estimate['mb']:.1f} MB)")
check("every source image in the fold is cached",
      len(train_ds._cache) == cache_estimate["images"],
      f"{len(train_ds._cache)} of {cache_estimate['images']}")
check("cached loading beats uncached",
      results["cached"][best] > results["uncached"][best],
      f"{results['cached'][best]:.1f} vs {results['uncached'][best]:.1f} tiles/s "
      f"at num_workers={best}")

warm_stats = cache_stats.get("warm_reload")
check("a second load of the same fold comes back warm",
      bool(warm_stats) and warm_stats["source"] == "warm",
      f"{warm_stats['source']} in {warm_stats['seconds']:.1f} s -- "
      f"{warm_stats['reason']}" if warm_stats else "cache_preload is off")
check("persistent cache file written for this fold",
      bool(warm_stats) and (cache_dir).is_dir()
      and any(cache_dir.glob(f"{FOLD}-train-*.npy")),
      f"{sorted(p.name for p in cache_dir.glob(f'{FOLD}-*.npy'))}")

check("dataloader.yaml written with a host-scoped entry",
      dataloader_path.is_file()
      and PATHS["platform"] in (yaml.safe_load(dataloader_path.read_text())
                                or {}).get("hosts", {}),
      f"hosts[{PATHS['platform']}].num_workers={best}")

failed = [n for n, ok in checks if not ok]
print(f"\n{len(checks) - len(failed)}/{len(checks)} checks passed")
if failed:
    raise AssertionError("failed checks: " + ", ".join(failed))

## Push the measured configuration

Only `configs/dataloader.yaml` is new here — the dataset itself is code, and
the manifests were pushed by step 3. `expect` names the file so that a push
which quietly changes nothing is diagnosed rather than reported as a no-op.

In [ ]:
from scripts.push_results import push_results

push_results(
    "step 4: dataset, synchronized augmentation, sampler",
    paths=PATHS,
    expect=[dataloader_path],
)